# MNIST workshop

You will train a small network to read handwritten digits, then reimplement the same idea yourself so the training loop is no longer a black box.

**What is already written**

1. `load_mnist()` loads and normalizes the images.
2. The outer loops of `train_classic_nn` and `accuracy`, and the checker and training loop for `ManualNet`.
3. `draw_and_predict()` opens a window where you can draw a digit.

**What you write**

1. `ClassicNN`: create the layers in `__init__`, then write `forward` (flatten, linear, sigmoid, linear, sigmoid).
2. The two checks in the "Check the shapes" cell.
3. The five steps inside `train_classic_nn`.
4. The counting inside `accuracy`.
5. `ManualNet`: the same network in NumPy, including the backward pass. The formulas are in section 3.

Every `# TODO` comes with hints. They point you in a direction and never give the line itself. Read them slowly, and try before you look anything up.

**How to work through the notebook**

- Run the cells from top to bottom. Read the note above a cell before you run it.
- A batch is a stack of images processed together. `B` is the number of images in that stack.
- An image starts as shape `(B, 1, 28, 28)`: batch, 1 gray channel, 28 rows, 28 columns.
- After training with learning rate 1, test accuracy should land well above 10% (a random guess). It is usually around 90% after 3 epochs.
- The drawing window needs a screen on the machine running the kernel. Run that last cell locally.

Install, if you do not already have these packages:

```
pip install torch torchvision numpy pillow matplotlib
```


In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import tkinter as tk
from PIL import Image, ImageDraw

# Same numbers torchvision uses for MNIST. The drawing tool must reuse them.
MNIST_MEAN = 0.1307
MNIST_STD = 0.3081

# Makes the MNIST download succeed on macOS Python installs that ship
# without the system certificate bundle. Verification stays on.
try:
    import ssl
    import certifi
    ssl._create_default_https_context = lambda: ssl.create_default_context(
        cafile=certifi.where()
    )
except ImportError:
    pass

torch.manual_seed(0)
np.random.seed(0)


## Shape cheat sheet

Nothing to fill in. Run the cell once.

Each line builds one layer on a fake batch of 4 images and prints the shape that comes out. These layers are thrown away afterwards. They are not the network you train later.

`Conv2d` is included so the name matches the slides. The network below does not use convolution. It only uses Flatten, Linear, and Sigmoid.


In [ ]:
x = torch.randn(4, 1, 28, 28)             # 4 MNIST images: (B, C, H, W)
print("image          :", tuple(x.shape))

# Flatten turns each image into one vector. start_dim=1 keeps the batch axis.
flatten = nn.Flatten(start_dim=1)
x_flat = flatten(x)                       # (4, 1, 28, 28) -> (4, 784)
print("Flatten        :", tuple(x_flat.shape))

# Linear: y = x @ W.T + b
# in_features is the length of one input vector (28*28 = 784).
# out_features is the number of neurons in this layer.
fc1 = nn.Linear(in_features=784, out_features=128)
h = fc1(x_flat)                           # (4, 784) -> (4, 128)
print("Linear 784->128:", tuple(h.shape))
print("   weight W    :", tuple(fc1.weight.shape), "(out_features, in_features)")
print("   bias b      :", tuple(fc1.bias.shape), "(out_features,)")

# Sigmoid squashes every number into (0, 1). The shape does not change.
# sigmoid(z) = 1 / (1 + exp(-z))
sigmoid = nn.Sigmoid()
a = sigmoid(h)                            # (4, 128) -> (4, 128)
print("Sigmoid        :", tuple(a.shape))

fc2 = nn.Linear(in_features=128, out_features=10)
out = fc2(a)                              # (4, 128) -> (4, 10), one score per digit
print("Linear 128->10 :", tuple(out.shape))

# Conv2d slides a small filter across the image.
# With kernel 3, padding 1 and stride 1, height and width stay 28.
conv = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, stride=1, padding=1)
c = conv(x)                               # (4, 1, 28, 28) -> (4, 8, 28, 28)
print("Conv2d 1->8    :", tuple(c.shape))
print("   weight      :", tuple(conv.weight.shape), "(out_channels, in_channels, k, k)")
print("   bias        :", tuple(conv.bias.shape), "(out_channels,)")
# H_out = (H + 2*padding - kernel_size) // stride + 1
# here: (28 + 2*1 - 3) // 1 + 1 = 28
# A Linear layer after a convolution needs Flatten first:
# (4, 8, 28, 28) -> (4, 8*28*28)

# backward fills .grad on every parameter that was used.
# It must start from a single number, so this demo uses a fake loss.
loss = out.mean()
loss.backward()
print(
    "backward       : fc1.weight.grad",
    tuple(fc1.weight.grad.shape),
    "(same shape as fc1.weight)",
)

# zero_grad before the next backward, otherwise gradients add up across batches.
fc1.zero_grad()
fc2.zero_grad()
print("zero_grad      : fc1.weight.grad =", fc1.weight.grad)  # None
# On a full model, model.zero_grad() clears every layer at once.


## 1. Load MNIST

MNIST is 70,000 grayscale digits, 28×28 pixels, labeled from 0 to 9.

- 60,000 images for training
- 10,000 images for testing (the network never trains on these)

`ToTensor()` turns a PIL image into a float tensor with pixels in `[0, 1]` and shape `(1, 28, 28)`.

`Normalize` then subtracts `0.1307` and divides by `0.3081`, so a typical pixel sits near 0 instead of near 0.5. The drawing tool in section 4 applies the same two numbers. If those numbers differ between training and prediction, the network is looking at a different kind of image than the one it learned.

A `DataLoader` groups images into batches. Training shuffles them so each epoch (one full pass over the training set) sees a new order. The test loader does not shuffle, so accuracy is comparable from run to run.


In [ ]:
def load_mnist(batch_size: int = 64, data_dir: str = "./data"):
    """Download MNIST if needed, normalize it, and return train and test loaders."""
    import os
    import time

    start = time.time()
    raw_dir = os.path.join(data_dir, "MNIST", "raw")
    if os.path.isdir(raw_dir) and len(os.listdir(raw_dir)) > 0:
        print(f"[1/4] Dataset already in {os.path.abspath(raw_dir)} (no download)")
    else:
        print(f"[1/4] Dataset missing. Downloading into {os.path.abspath(data_dir)} ...")

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((MNIST_MEAN,), (MNIST_STD,)),
    ])

    print("[2/4] Loading the training set (60,000 images) ...")
    train_set = datasets.MNIST(root=data_dir, train=True, download=True, transform=transform)
    print(f"      -> OK, {len(train_set)} images ({time.time() - start:.1f}s)")

    print("[3/4] Loading the test set (10,000 images) ...")
    test_set = datasets.MNIST(root=data_dir, train=False, download=True, transform=transform)
    print(f"      -> OK, {len(test_set)} images ({time.time() - start:.1f}s)")

    print(f"[4/4] Building DataLoaders (batch_size={batch_size}) ...")
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)
    print(f"      -> OK: {len(train_loader)} training batches, {len(test_loader)} test batches")
    print(f"Done in {time.time() - start:.1f}s")
    return train_loader, test_loader


### Look at a few digits

Run this once so the pictures are not abstract. The loader returns normalized tensors, which look washed out if you draw them directly. The cell undoes the normalization for display only: `pixel * std + mean`.


In [ ]:
import matplotlib.pyplot as plt

train_loader, test_loader = load_mnist(batch_size=64)

images, labels = next(iter(train_loader))
shown = images[:8] * MNIST_STD + MNIST_MEAN

fig, axes = plt.subplots(2, 4, figsize=(8, 4))
for ax, img, label in zip(axes.ravel(), shown, labels[:8]):
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(f"label {int(label)}")
    ax.axis("off")
fig.suptitle("Eight training images, after undoing the normalization")
fig.tight_layout()
plt.show()

print("one batch of images:", tuple(images.shape), "  labels:", tuple(labels.shape))


## 2. The PyTorch network

You create the layers in `__init__` and write `forward`.

```
(B, 1, 28, 28)   image
      | Flatten
(B, 784)
      | Linear + Sigmoid
(B, 32)          hidden layer
      | Linear + Sigmoid
(B, 10)          one score per digit, each between 0 and 1
      | argmax
predicted digit
```

This is the lecture network: a sigmoid after the hidden layer and a sigmoid after the output layer. `ManualNet` in section 3 is the same network, written in NumPy.

The ten scores do not need to add up to 1. The predicted digit is the index of the largest score (`argmax`).

Training uses mean squared error against a **one-hot** target. The label `3` becomes `[0, 0, 0, 1, 0, 0, 0, 0, 0, 0]`. MSE pushes the matching score toward 1 and the other nine toward 0.


In [ ]:
class ClassicNN(nn.Module):
    def __init__(self, input_size=784, hidden_size=32, output_size=10):
        super().__init__()
        # TODO: create the building blocks of the network and store each one on `self`.
        #
        # Look at the diagram in the note above: which layers, in which order?
        # The "Shape cheat sheet" cell already builds every layer type you need.
        # Reuse the idea, but not the sizes.
        #
        # Questions to ask yourself:
        #   - For each Linear layer: how long is ONE input vector, and how many neurons
        #     should the layer have? Use the constructor arguments (input_size,
        #     hidden_size, output_size) instead of typing numbers, so the class
        #     works for any size.
        #   - Sigmoid has no size at all. Does it hold any weights? Do you need two
        #     separate objects, or could one be reused? (Both work. Pick one and stay
        #     consistent in forward.)
        #   - Store each layer as an attribute (self.<name> = ...). PyTorch only tracks
        #     the weights of layers that are attributes of the model.
        #     Choose names you will remember, because you need them in forward.
        pass

    def forward(self, x):
        # TODO: send x through the layers you created, in the order of the diagram.
        #
        #   - A layer object can be applied to a tensor as if it were a function.
        #     The cheat sheet does exactly this with fc1(x_flat).
        #   - The output of one step is the input of the next step.
        #   - Fill in the expected shape after each step on paper before you code:
        #       (B, 1, 28, 28) -> (B, ?) -> (B, ?) -> (B, ?) -> (B, ?) -> (B, ?)
        #     If a shape surprises you, print x.shape between two steps.
        #   - Do not forget to return the result. Delete the raise below once you do.
        raise NotImplementedError("Write ClassicNN.forward")


### Check the shapes before any training

Run this after `forward` is written. A wrong layer size fails here, in one line, instead of halfway through an epoch. Each score must also lie strictly between 0 and 1, which is what the output sigmoid does.


In [ ]:
_probe = ClassicNN()
_scores = _probe(torch.zeros(2, 1, 28, 28))
# TODO: check two facts about _scores with `assert` statements.
# Give each assert a short message that tells the reader what went wrong.
#
#   1. The shape. Two images go in and there are ten digit classes.
#      What shape should come out? Compare it with the shape of _scores.
#      (Tip: tuple(t.shape) gives something easy to compare and to print.)
#
#   2. The range. Every score should lie strictly between 0 and 1.
#      A comparison such as `t > 0` returns a tensor of True/False, one per entry.
#      How do you combine two such tensors ("and")? How do you ask
#      "is it True EVERYWHERE?" Look up torch.all in the PyTorch docs.
#
# If an assert fails, its message should tell you which part of ClassicNN to revisit.
print("Shape and sigmoid range look right.")


### The training loop

The loop, the one-hot targets, and the print are already written. You write the five steps inside the batch loop, in this order:

1. `zero_grad` clears the gradients left by the previous batch. Without this, gradients accumulate.
2. `forward` turns the images into 10 scores.
3. The loss compares those scores with the one-hot labels. One number, smaller is better.
4. `backward` applies the chain rule and writes a gradient into every weight and bias.
5. The update `parameter -= learning_rate * gradient` takes a step downhill. It is done by hand here, instead of `torch.optim`, so the rule is visible. Put that update inside `torch.no_grad()`.

Delete the `raise` once those five steps are in place. The lines under it record the loss for the plot in section 5.

`model.train()` matters when a network has dropout or batch-norm. This one does not. The call is still the right habit.

`epochs=3` and `lr=1` are enough for this network. PyTorch averages the loss over the batch, which is why this learning rate is larger than the `0.1` used later in `ManualNet`. If you change `hidden_size`, build a new model and train again. The old weights belong to the old size.


In [ ]:
def train_classic_nn(model, train_loader, epochs=3, lr=1.0, device="cpu"):
    model.to(device)
    model.train()
    loss_history = []

    for epoch in range(epochs):
        total_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            targets = torch.nn.functional.one_hot(labels, num_classes=10).float()

            # TODO: write the five steps of one training iteration, in this order.
            # The variable holding the loss MUST be called `loss`, because the line
            # after the raise uses it.
            #
            # Step 1. Clear old gradients.
            #   The parameters still remember the gradients of the previous batch.
            #   Which method of the model resets them? (The cheat sheet used it on
            #   single layers.) What would happen to the gradients if you skipped it?
            #
            # Step 2. Forward pass.
            #   Apply the model to `images`, the same way the cheat sheet applied
            #   fc1 to x_flat. Call the result `scores`.
            #   Its shape should match `targets`. Check it if unsure.
            #
            # Step 3. Loss.
            #   Read the name "mean squared error" from right to left: take the error
            #   between scores and targets, square it, then average it. You can build
            #   this from basic tensor operations or find a ready-made version in
            #   torch.nn.functional. The result must be ONE number.
            #
            # Step 4. Backward pass.
            #   One call on `loss` fills the .grad of every parameter. The cheat sheet
            #   shows which one. Only do this AFTER step 1. Why does the order matter?
            #
            # Step 5. Gradient descent update, by hand.
            #   The rule is in the markdown above: every parameter takes a step against
            #   its gradient, and the step size is `lr`.
            #   You need:
            #     - a way to visit ALL the weights and biases of the model
            #       (look at the parameters() method of nn.Module)
            #     - an update that changes each tensor IN PLACE, otherwise the model
            #       itself never changes
            #     - torch.no_grad() around the update, so that PyTorch does not record
            #       the update itself as part of the computation
            #
            # Debug tip: the printed mean loss should go down from epoch to epoch.
            # If it stays flat, re-check steps 1, 4 and 5 first.

            raise NotImplementedError("Implement the 5 training steps inside train_classic_nn")

            total_loss += loss.item()

        mean_loss = total_loss / len(train_loader)
        loss_history.append(mean_loss)
        print(f"Epoch {epoch + 1}/{epochs} - mean training loss: {mean_loss:.4f}")

    return loss_history


### Accuracy

Accuracy is the fraction of images whose predicted digit matches the label.

The network is switched to `eval()` and wrapped in `no_grad()` because this pass must not change any weight. `argmax(dim=1)` picks a digit for each row of scores.

A model that ignores the image and guesses scores about 10%. That is the number you should beat.

The loop over batches is written. You write the counting inside it.


In [ ]:
def accuracy(model, loader, device="cpu"):
    """Fraction of correct digits on a DataLoader. Multiply by 100 for a percent."""
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            # TODO: update `correct` and `total` for this batch.
            #
            #   1. Get the scores of the model for `images`. Shape (B, 10).
            #   2. Turn each row of 10 scores into ONE predicted digit.
            #      Which position in the row holds the largest score? The markdown above
            #      names the function. Think about which dim to reduce over:
            #      the result should have shape (B,), one digit per image.
            #   3. Compare the predictions with `labels`. An element-wise comparison
            #      gives a tensor of True/False, one per image.
            #   4. Count the Trues. Adding booleans counts them, but the result is still
            #      a tensor: convert it to a plain Python number before adding it
            #      to `correct` (look at .item()).
            #   5. Increase `total` by the number of images in THIS batch.
            #      Do not type 64: the last batch of an epoch is smaller.
            pass

    return correct / total if total > 0 else 0.0


## 3. The same network, written by hand

This is the part you implement. `ManualNet` does not use `nn.Linear` or `loss.backward()`. You store the weights yourself and apply the lecture formulas.

Use the same layout as the lecture: each example is a **row**.

| name | shape | role |
| --- | --- | --- |
| `X` | `(B, 784)` | flattened, normalized images |
| `W1` | `(784, hidden)` | first-layer weights |
| `b1` | `(1, hidden)` | first-layer biases |
| `A1` | `(B, hidden)` | hidden activations, after the sigmoid |
| `W2` | `(hidden, 10)` | second-layer weights |
| `b2` | `(1, 10)` | second-layer biases |
| `yhat` | `(B, 10)` | output activations, after the sigmoid |

PyTorch stores a `Linear` weight as `(out_features, in_features)` and computes `x @ W.T`. This NumPy network stores each weight as `(in, out)` and computes `X @ W`, the same layout as the lecture. Build the matrices from the table above, entry by entry.

**`__init__`.** Draw `W1` and `W2` from a normal distribution with standard deviation `0.01`. Start `b1` and `b2` at 0. Large random weights saturate the sigmoid immediately, and then almost nothing learns.

**`forward(X)`.** If `X` still has image dimensions, flatten it to `(B, 784)`. Then

$$
Z_1 = X W_1 + b_1, \qquad A_1 = \sigma(Z_1)
$$

$$
Z_2 = A_1 W_2 + b_2, \qquad \hat{y} = \sigma(Z_2)
$$

`sigmoid` is already defined in the next cell. Return `\hat{y}`, shape `(B, 10)`, every entry between 0 and 1.

Save `X`, `A1`, and `yhat` on `self`. `backward` reads that saved forward pass. This is the same network as `ClassicNN`: a sigmoid on the hidden layer and a sigmoid on the output. The predicted digit is the index of the largest score.

**`backward(y_true, learning_rate)`.** `y_true` is one-hot with the same shape as `yhat`. For mean squared error through a sigmoid, the output error is

$$
\delta^{(\mathrm{out})} = (\hat{y} - y)\, \hat{y}\,(1 - \hat{y})
$$

That product is the gap `\hat{y} - y` times the slope of the output sigmoid. Push it back to the hidden layer. `\odot` is entry-by-entry multiplication, not a matrix product:

$$
\delta^{(\mathrm{hidden})} = \big(\delta^{(\mathrm{out})} W_2^{\top}\big) \odot A_1\,(1 - A_1)
$$

The gradients are "what entered the layer" times "the delta of the layer". On a batch, the bias gradient is the sum of the deltas over the rows:

$$
dW_2 = A_1^{\top} \delta^{(\mathrm{out})}, \qquad db_2 = \sum_{\mathrm{rows}} \delta^{(\mathrm{out})}
$$

$$
dW_1 = X^{\top} \delta^{(\mathrm{hidden})}, \qquad db_1 = \sum_{\mathrm{rows}} \delta^{(\mathrm{hidden})}
$$

Compute every gradient **before** changing any weight. `delta_hidden` still needs the current `W2`. Then

```
W1 -= learning_rate * dW1
```

and the same for `b1`, `W2`, and `b2`.

These products sum over the batch, so do not also divide by the batch size. With that choice, `learning_rate=0.1` is a good start. If you divide the deltas by `B`, raise the learning rate or the steps become tiny. This `0.1` is for `ManualNet`. `train_classic_nn` uses `1`, because PyTorch averages the loss and the step size is not the same.

**`predict(X)`.** Run `forward` and return the class index of each row, shape `(B,)`.

**Checks, once the four methods exist**

- `forward` on two images returns shape `(2, 10)` and values strictly between 0 and 1.
- One call to `backward` changes `W2`. If it does not, the gradients were never applied.
- After one or two epochs, test accuracy should be above 90% with `hidden_size=32` and `learning_rate=0.1`. A number stuck near 10% means the weight update is not applied.
- Compute all four gradients first, then update. Updating `W2` early uses the new matrix in `delta_hidden`.


In [ ]:
def sigmoid(z):
    """1 / (1 + exp(-z)). The clip avoids overflow for large z."""
    return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))


class ManualNet:
    """Two sigmoid layers in NumPy. Fill in the four methods."""

    def __init__(self, input_size=784, hidden_size=32, output_size=10):
        # TODO: W1 (input_size, hidden_size), b1 (1, hidden_size)
        #       W2 (hidden_size, output_size), b2 (1, output_size)
        # Weights ~ Normal(0, 0.01). Biases = 0.
        self.W1 = None
        self.b1 = None
        self.W2 = None
        self.b2 = None

    def forward(self, X):
        # TODO: flatten if needed, two linear+sigmoid steps, save X, A1, yhat.
        # Return yhat with shape (batch, output_size).
        return None

    def backward(self, y_true, learning_rate=0.1):
        # TODO: deltas from the formulas above, then the four updates.
        # Use the W2 from before the update when you form delta_hidden.
        return None

    def predict(self, X):
        # TODO: class index per row, shape (batch,).
        return None


### Train your NumPy network

The loop below is the same five steps as `train_classic_nn`, calling your methods instead of PyTorch. It does not train anything until `ManualNet` is filled in. Run it again after you write the four methods.

Before the full training run, the cell fits **one batch for 20 updates**. The loss on that batch should fall. If it does not, fix `backward` before waiting through three epochs.

Images coming out of the loader are already normalized. The loop only converts them to NumPy and builds the one-hot targets.


In [ ]:
def _manual_net_ready(train_loader):
    """Return a fresh ManualNet if the methods look usable and one batch learns, else None."""
    probe = ManualNet()
    if probe.W1 is None or probe.W2 is None:
        print("ManualNet is not ready: weights are still None. Fill in __init__, then run this cell again.")
        return None
    x = np.zeros((2, 784), dtype=np.float64)
    y = np.zeros((2, 10), dtype=np.float64)
    y[:, 3] = 1.0
    try:
        yhat = probe.forward(x)
    except Exception as exc:
        print(f"forward raised {type(exc).__name__}: {exc}")
        return None
    if yhat is None or getattr(yhat, "shape", None) != (2, 10):
        print(f"forward should return shape (2, 10). Got {None if yhat is None else getattr(yhat, 'shape', None)}.")
        return None
    if not np.all((yhat > 0) & (yhat < 1)):
        print("forward should be a sigmoid on the output, so every score is strictly between 0 and 1.")
        return None
    before = np.array(probe.W2, copy=True)
    try:
        probe.backward(y, learning_rate=0.1)
    except Exception as exc:
        print(f"backward raised {type(exc).__name__}: {exc}")
        return None
    if probe.W2 is None or np.allclose(before, probe.W2):
        print("backward did not change W2. Compute dW2 and apply W2 -= learning_rate * dW2.")
        return None
    try:
        pred = probe.predict(x)
    except Exception as exc:
        print(f"predict raised {type(exc).__name__}: {exc}")
        return None
    if pred is None or np.asarray(pred).shape != (2,):
        print(f"predict should return shape (2,). Got {None if pred is None else np.asarray(pred).shape}.")
        return None

    print("ManualNet passed the shape checks.")
    images, labels = next(iter(train_loader))
    x = images.numpy().reshape(images.shape[0], -1)
    y = np.eye(10, dtype=np.float64)[labels.numpy()]
    probe = ManualNet()
    first = float(np.mean((probe.forward(x) - y) ** 2))
    for _ in range(20):
        probe.forward(x)
        probe.backward(y, learning_rate=0.1)
    last = float(np.mean((probe.forward(x) - y) ** 2))
    print(f"One batch, 20 updates: loss {first:.4f} -> {last:.4f}")
    if not last < first:
        print("The loss on this batch did not fall. Check the deltas, and update every weight only after all four gradients are computed.")
        return None
    print("Loss fell on one batch. Training a fresh network on the full training set.")
    return ManualNet()


def train_manual_net(net, train_loader, test_loader, epochs=3, lr=0.1):
    for epoch in range(epochs):
        total_loss, n_batches = 0.0, 0
        for images, labels in train_loader:
            x = images.numpy().reshape(images.shape[0], -1)
            y = np.eye(10, dtype=np.float64)[labels.numpy()]
            yhat = net.forward(x)
            total_loss += float(np.mean((yhat - y) ** 2))
            net.backward(y, learning_rate=lr)
            n_batches += 1
        correct = total = 0
        for images, labels in test_loader:
            x = images.numpy()
            pred = np.asarray(net.predict(x))
            correct += int((pred == labels.numpy()).sum())
            total += len(labels)
        print(
            f"Epoch {epoch + 1}/{epochs} - mean loss: {total_loss / n_batches:.4f}"
            f" - test accuracy: {100.0 * correct / total:.1f}%"
        )


manual_net = _manual_net_ready(train_loader)
if manual_net is not None:
    train_manual_net(manual_net, train_loader, test_loader, epochs=3, lr=0.1)


## 4. Draw a digit

This part is already written. It opens a black window. Draw a white digit with the mouse, then press **Predict**.

MNIST digits are not full-frame drawings. They are a small, roughly centered gray blob. `preprocess` copies that recipe:

1. Crop away the empty black border.
2. Resize so the longest side is 20 pixels.
3. Paste the result into the middle of a black 28×28 image.
4. Shift the image so the center of mass lands on pixel `(14, 14)`.
5. Scale pixels from `0..255` down to `0..1`.

`draw_and_predict` then applies the same mean and standard deviation as `load_mnist`, and asks the network for the argmax score. The printed score is the sigmoid output for the chosen digit, between 0 and 1. A value near 1 means MSE pushed that score toward the one-hot target. The ten scores do not have to add up to 1.


In [ ]:
def preprocess(image):
    """
    Turn a drawing (280x280, white digit on black) into a 28x28 image
    in the MNIST style: cropped, longest side 20, centered.
    Returns None when the canvas is empty.
    """
    arr = np.array(image)
    ys, xs = np.nonzero(arr > 20)  # lit pixels
    if len(xs) == 0:
        return None

    # 1. Crop around the digit.
    digit = image.crop((xs.min(), ys.min(), xs.max() + 1, ys.max() + 1))

    # 2. Longest side becomes 20 pixels.
    resample = Image.Resampling.LANCZOS if hasattr(Image, "Resampling") else Image.LANCZOS
    w, h = digit.size
    scale = 20.0 / max(w, h)
    digit = digit.resize((max(1, round(w * scale)), max(1, round(h * scale))), resample)

    # 3. Paste into the middle of a black 28x28 image.
    img28 = Image.new("L", (28, 28), 0)
    img28.paste(digit, ((28 - digit.size[0]) // 2, (28 - digit.size[1]) // 2))

    # 4. Shift so the center of mass is at (14, 14).
    a = np.array(img28).astype(np.float32)
    total = a.sum()
    cy = (a.sum(axis=1) * np.arange(28)).sum() / total
    cx = (a.sum(axis=0) * np.arange(28)).sum() / total
    a = np.roll(a, (round(14 - cy), round(14 - cx)), axis=(0, 1))

    return a / 255.0  # pixels in [0, 1]


def draw_and_predict(model, device="cpu"):
    """
    Open a window, draw a digit, and press Predict.
    The drawing is cropped, centered, normalized, and passed to the model.
    """
    canvas_size = 280
    brush_size = 12
    model.eval()

    window = tk.Tk()
    window.title("Draw a digit (0-9)")

    canvas = tk.Canvas(window, width=canvas_size, height=canvas_size, bg="black")
    canvas.pack()

    image = Image.new("L", (canvas_size, canvas_size), color=0)
    draw_ctx = ImageDraw.Draw(image)

    result_label = tk.Label(
        window,
        text="Draw a digit, then click Predict",
        font=("Arial", 14),
    )
    result_label.pack()

    def on_draw(event):
        x, y = event.x, event.y
        r = brush_size
        canvas.create_oval(x - r, y - r, x + r, y + r, fill="white", outline="white")
        draw_ctx.ellipse([x - r, y - r, x + r, y + r], fill=255)

    def clear_canvas():
        canvas.delete("all")
        draw_ctx.rectangle([0, 0, canvas_size, canvas_size], fill=0)
        result_label.config(text="Draw a digit, then click Predict")

    def predict_digit():
        arr = preprocess(image)  # (28, 28), values in [0, 1]
        if arr is None:
            result_label.config(text="Draw a digit first")
            return
        arr = (arr - MNIST_MEAN) / MNIST_STD  # same normalization as training

        tensor = torch.tensor(arr, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)

        with torch.no_grad():
            scores = model(tensor)                  # (1, 10)
            predicted = torch.argmax(scores, dim=1).item()
            score = scores[0, predicted].item()

        result_label.config(text=f"Predicted digit: {predicted}  (score: {score:.2f})")

    canvas.bind("<B1-Motion>", on_draw)

    button_frame = tk.Frame(window)
    button_frame.pack(pady=10)
    tk.Button(button_frame, text="Predict", command=predict_digit).pack(side="left", padx=5)
    tk.Button(button_frame, text="Clear", command=clear_canvas).pack(side="left", padx=5)

    window.mainloop()


## 5. Train `ClassicNN` and measure it

Run this after `forward` and the five training steps are written. It trains for 3 epochs with learning rate 1. On CPU that usually takes under a minute. Test accuracy should land near 90%.

The cell then picks one test image and prints the predicted digit next to the true label. That is the same path the drawing window uses: normalize, forward, argmax.


In [ ]:
device = "cpu"
model = ClassicNN()
train_classic_nn(model, train_loader, epochs=3, lr=1.0, device=device)

test_acc = accuracy(model, test_loader, device)
print(f"Test accuracy: {test_acc * 100:.1f}%")

images, labels = next(iter(test_loader))
with torch.no_grad():
    scores = model(images.to(device))
    pred = scores.argmax(dim=1)

index = 0
print(f"Example: predicted {int(pred[index])}, label {int(labels[index])}")
print("scores for that image:", [round(v, 2) for v in scores[index].tolist()])


### Digits the network gets wrong

Accuracy hides the interesting mistakes. This cell keeps 12 test images the trained model mislabels, and prints the true digit next to the prediction. Pairs that show up often, such as 4 and 9 or 3 and 5, are the ones to try in the drawing window.


In [ ]:
model.eval()
wrong_images, wrong_true, wrong_pred = [], [], []
with torch.no_grad():
    for images, labels in test_loader:
        pred = model(images.to(device)).argmax(dim=1).cpu()
        mismatch = pred != labels
        for img, y, p in zip(images[mismatch], labels[mismatch], pred[mismatch]):
            wrong_images.append(img)
            wrong_true.append(int(y))
            wrong_pred.append(int(p))
            if len(wrong_images) == 12:
                break
        if len(wrong_images) == 12:
            break

print(f"Showing {len(wrong_images)} mistakes from the test set.")
fig, axes = plt.subplots(3, 4, figsize=(8, 6))
for ax, img, y, p in zip(axes.ravel(), wrong_images, wrong_true, wrong_pred):
    ax.imshow((img.squeeze() * MNIST_STD + MNIST_MEAN), cmap="gray")
    ax.set_title(f"true {y}, pred {p}")
    ax.axis("off")
for ax in axes.ravel()[len(wrong_images):]:
    ax.axis("off")
fig.tight_layout()
plt.show()


### Three learning rates

Train three fresh networks for 3 epochs. They start from the same weights (`torch.manual_seed(0)`), with three step sizes:

- `0.001` is too small. The loss barely moves, and accuracy stays near a random guess.
- `1` is the step size from the cell above. The loss keeps falling.
- `50` is too large. The loss drops once, then sticks near `0.1`. That is the loss of a flat wrong answer: one target entry is 1, the other nine are 0, and the mean of those squared errors is `0.1`.

The plot compares the three training curves. The drawing window still uses the model from the first training cell, not these three.


In [ ]:
lr_histories = {}
for lr in (0.001, 1.0, 50.0):
    torch.manual_seed(0)
    net = ClassicNN()
    print()
    print(f"Learning rate {lr}")
    lr_histories[lr] = train_classic_nn(net, train_loader, epochs=3, lr=lr, device=device)

fig, ax = plt.subplots(figsize=(6, 4))
for lr, hist in lr_histories.items():
    ax.plot(range(1, len(hist) + 1), hist, marker="o", label=f"lr={lr}")
ax.set_xlabel("epoch")
ax.set_ylabel("mean training loss")
ax.set_xticks([1, 2, 3])
ax.legend()
fig.tight_layout()
plt.show()


### Draw your own digit

Run this after the training cell. A window opens. Close it to get the kernel back.

Try a digit from the mistake grid above, drawn the way MNIST does: a single stroke, not too thick, not touching the border, roughly in the middle. The network only saw digits that look like that.


In [ ]:
# Opens a window. Close it when you are done.
draw_and_predict(model, device=device)
